In [1]:
import os
import re
import json
import uuid
import hashlib
import time
from typing import Literal
from collections import Counter
from dotenv import load_dotenv
from pydantic import BaseModel
from tqdm import tqdm

from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import DocItemLabel

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')

openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✓ Imports done")
print("✓ Environment loaded")

✓ Imports done
✓ Environment loaded


In [2]:
FILINGS = [
    {
        "company": "Apple",
        "ticker": "AAPL",
        "exchange": "NASDAQ",
        "filing_type": "10-Q",
        "period": "Q2-FY2026",
        "currency": "USD",
        "accounting_standard": "US GAAP",
        "local_path": "../data/raw/aapl_10q_q2_2026.pdf"
    },
    {
        "company": "Microsoft",
        "ticker": "MSFT",
        "exchange": "NASDAQ",
        "filing_type": "10-Q",
        "period": "Q1-FY2026",
        "currency": "USD",
        "accounting_standard": "US GAAP",
        "local_path": "../data/raw/msft_10q_q1_2026.pdf"
    }
]

print(f"Filings registered: {len(FILINGS)}")
for f in FILINGS:
    path_exists = os.path.exists(f["local_path"])
    print(f"  {f['ticker']} | {f['period']} | file found: {path_exists}")

Filings registered: 2
  AAPL | Q2-FY2026 | file found: True
  MSFT | Q1-FY2026 | file found: True


In [3]:
converter = DocumentConverter()
all_results = {}

for filing in FILINGS:
    print(f"Converting {filing['company']}... (takes 2-3 mins)")
    result = converter.convert(filing["local_path"])
    all_results[filing["ticker"]] = {
        "result": result,
        "metadata": {
            "company": filing["company"],
            "ticker": filing["ticker"],
            "period": filing["period"],
            "currency": filing["currency"]
        }
    }
    print(f"  ✓ {filing['ticker']} converted")

print("\nAll documents converted.")

Converting Apple... (takes 2-3 mins)


[INFO] 2026-07-08 13:55:48,759 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-08 13:55:48,780 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-08 13:55:48,823 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\aksaj\Documents\ai-engineering\financial-reconciliation-agent\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-08 13:55:48,824 [RapidOCR] main.py:50: Using C:\Users\aksaj\Documents\ai-engineering\financial-reconciliation-agent\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-08 13:55:49,323 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-08 13:55:49,325 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-08 13:55:49,332 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\aksaj\Documents\ai-engineering\financial-reconciliation-agent\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-07-08 13:

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

  ✓ AAPL converted
Converting Microsoft... (takes 2-3 mins)
  ✓ MSFT converted

All documents converted.


In [4]:
# Regex patterns — deterministic fast path
SECTION_PATTERNS = {
    "Income Statement": [
        "statements of operations",
        "statements of income",
        "income statements",
        "income statement",
        "comprehensive income"
    ],
    "Balance Sheet": [
        "balance sheet",
        "financial position",
        "balance sheets"
    ],
    "Cash Flow": [
        "cash flows",
        "cash flow statement",
        "cash flows statements"
    ],
    "Shareholders Equity": [
        "shareholders equity",
        "stockholders equity",
        "stockholders' equity statements",
        "equity statements"
    ],
    "MD&A": [
        "management's discussion",
        "results of operations",
        "summary results of operations",
        "item 2. management",
        "item 2.    management"
    ],
    "Risk Factors": [
        "risk factors",
        "item 1a",
        "item 1a.",
        "strategic and competitive risks"
    ],
    "Legal Proceedings": [
        "legal proceedings",
        "item 1. legal",
        "legal proceeding"
    ],
    "Notes": [
        "notes to condensed",
        "notes to consolidated",
        "notes to financial statements"
    ],
    "Signature": ["pursuant to the requirements", "duly authorized"],
    "Exhibits": ["exhibit 31", "exhibit 32", "incorporated by reference"]
}

def detect_section_regex(text: str) -> str:
    """Fast regex-based section detection."""
    text_lower = text.lower()
    for section, patterns in SECTION_PATTERNS.items():
        for pattern in patterns:
            if pattern in text_lower:
                return section
    return "General"


# Keep LLM classifier for unknown headers only
class SectionClassification(BaseModel):
    section: Literal[
        "Income Statement", "Balance Sheet", "Cash Flow",
        "Shareholders Equity", "MD&A", "Risk Factors",
        "Legal Proceedings", "Notes", "Signature", "Exhibits", "General"
    ]
    confidence: Literal["high", "low"]

section_cache = {}

def detect_section_llm(header_text: str) -> str:
    """LLM fallback for headers not matched by regex."""
    if header_text in section_cache:
        return section_cache[header_text]

    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """You are a financial document section classifier for SEC 10-Q filings.
Classify ONLY top-level major section headers.
Sub-headers, note titles, and navigation text → General, low confidence.

High confidence examples:
  "INCOME STATEMENTS" → Income Statement
  "BALANCE SHEETS" → Balance Sheet
  "STOCKHOLDERS' EQUITY STATEMENTS" → Shareholders Equity
  "NOTES TO FINANCIAL STATEMENTS" → Notes
  "CASH FLOWS STATEMENTS" → Cash Flow
  "MANAGEMENT'S DISCUSSION AND ANALYSIS" → MD&A
  "ITEM 1A. RISK FACTORS" → Risk Factors
  "ITEM 1. LEGAL PROCEEDINGS" → Legal Proceedings

Low confidence (General) examples:
  "Note 2 - Revenue", "Trade Receivables", "PART I Item 1"
  "Effective Tax Rate", "Accounting Principles" → all General"""
            },
            {"role": "user", "content": f"Classify: '{header_text}'"}
        ],
        response_format=SectionClassification,
        temperature=0
    )

    result = response.choices[0].message.parsed
    final = result.section if result.confidence == "high" else "General"
    section_cache[header_text] = final
    return final


def detect_section(header_text: str) -> str:
    """
    Hybrid section detection:
    1. Regex fast path — deterministic, free
    2. LLM fallback — for unknown headers only
    """
    regex_result = detect_section_regex(header_text)
    if regex_result != "General":
        return regex_result
    return detect_section_llm(header_text)


# Test
test_headers = [
    "INCOME STATEMENTS",
    "CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS",
    "BALANCE SHEETS",
    "STOCKHOLDERS' EQUITY STATEMENTS",
    "NOTES TO FINANCIAL STATEMENTS",
    "CASH FLOWS STATEMENTS",
    "MANAGEMENT'S DISCUSSION AND ANALYSIS",
    "ITEM 1A. RISK FACTORS",
    "ITEM 1. LEGAL PROCEEDINGS",
    "Note 2 - Revenue",
    "PART I Item 1",
    "Effective Tax Rate"
]

print("Testing hybrid classifier:\n")
for header in test_headers:
    result = detect_section(header)
    print(f"  [{result:25}] '{header}'")

Testing hybrid classifier:

  [Income Statement         ] 'INCOME STATEMENTS'
  [Income Statement         ] 'CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS'
  [Balance Sheet            ] 'BALANCE SHEETS'
  [Shareholders Equity      ] 'STOCKHOLDERS' EQUITY STATEMENTS'
  [Notes                    ] 'NOTES TO FINANCIAL STATEMENTS'
  [Cash Flow                ] 'CASH FLOWS STATEMENTS'
  [MD&A                     ] 'MANAGEMENT'S DISCUSSION AND ANALYSIS'
  [Risk Factors             ] 'ITEM 1A. RISK FACTORS'
  [Legal Proceedings        ] 'ITEM 1. LEGAL PROCEEDINGS'
  [General                  ] 'Note 2 - Revenue'
  [General                  ] 'PART I Item 1'
  [General                  ] 'Effective Tax Rate'


In [5]:
def get_page_no(element) -> int:
    """Extract page number from Docling element provenance."""
    try:
        return element.prov[0].page_no
    except:
        return None


def split_into_sentences(text: str) -> list:
    """Split prose into sentences for child chunks."""
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def is_meaningful_row(row: str) -> bool:
    """
    Filter out non-data rows from tables.
    Keeps only rows with actual financial figures.
    """
    cells = [c.strip() for c in row.split('|') if c.strip()]

    if not cells:
        return False

    # Must have at least 2 non-empty cells
    non_empty = [c for c in cells if c and c != '-']
    if len(non_empty) < 2:
        return False

    # Filter date header rows
    date_patterns = [
        'march', 'june', 'september', 'december',
        'january', 'february', 'april', 'july',
        'august', 'october', 'november'
    ]
    if any(month in row.lower() for month in date_patterns):
        return False

    # Must contain financial data — $ sign or comma-formatted number
    has_financial = any(
        '$' in cell or
        (any(char.isdigit() for char in cell) and ',' in cell)
        for cell in cells
    )
    return has_financial


def split_table_into_rows(table_markdown: str) -> list:
    """
    Split markdown table into individual row children.
    Each child = column header + separator + one data row.
    """
    lines = table_markdown.strip().split('\n')
    header = None
    separator = None
    data_rows = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if re.match(r'^\|[-|\s:]+\|$', line):
            separator = line
            continue
        if header is None:
            header = line
            continue
        if line.startswith('|'):
            data_rows.append(line)

    children = []
    for row in data_rows:
        if not is_meaningful_row(row):
            continue
        child_content = f"{header}\n{separator}\n{row}" if separator else f"{header}\n{row}"
        children.append(child_content)

    return children


def make_chunk_id(ticker: str, index: int) -> str:
    """Generate deterministic chunk ID for Pinecone."""
    raw = f"{ticker}_{index}"
    return hashlib.md5(raw.encode()).hexdigest()[:16]



In [6]:
    SECTION_BOUNDARY_PHRASES = [
        "item 1a", "risk factor",
        "item 1. legal", "legal proceeding",
        "the company is subject to various legal",
        "strategic and competitive risks",  # MSFT risk factors opener
        "part ii. other", "part ii other",
        "this item and other sections of this quarterly report",
    ]

In [7]:
# Module level — accessible everywhere
SECTION_BOUNDARY_PHRASES = [
    "item 1a", "risk factor",
    "item 1. legal", "legal proceeding",
    "the company is subject to various legal",
    "strategic and competitive risks",
    "this item and other sections of this quarterly report",
    "part ii. other", "part ii other"
]

HARDCODED_BOUNDARIES = {
    "this item and other sections of this quarterly report": "Risk Factors",
    "this report includes estimates, projections, statements relating": "Risk Factors",
}


def create_parent_child_chunks(ticker: str, result, filing_metadata: dict):
    parents = {}
    children = []
    current_section = "General"
    doc = result.document
    child_index = 0

    for element, level in doc.iterate_items():
        page_no = get_page_no(element)

        # Skip page headers and footers
        if hasattr(element, 'label') and element.label in [
            DocItemLabel.PAGE_HEADER,
            DocItemLabel.PAGE_FOOTER
        ]:
            continue

        # Section detection from SECTION_HEADER and TITLE elements
        if hasattr(element, 'label') and element.label in [
            DocItemLabel.SECTION_HEADER,
            DocItemLabel.TITLE
        ]:
            if hasattr(element, 'text'):
                detected = detect_section(element.text)
                if detected != "General":
                    current_section = detected
            continue

        # TABLE
        if hasattr(element, 'data'):
            table_text = element.export_to_markdown(doc=doc)
            if not table_text.strip():
                continue

            parent_id = str(uuid.uuid4())
            parents[parent_id] = {
                "content": table_text,
                "metadata": {
                    **filing_metadata,
                    "chunk_type": "table",
                    "section": current_section,
                    "page": page_no
                }
            }

            row_children = split_table_into_rows(table_text)
            for row in row_children:
                children.append({
                    "content": row,
                    "metadata": {
                        **filing_metadata,
                        "parent_id": parent_id,
                        "chunk_type": "table",
                        "section": current_section,
                        "page": page_no,
                        "child_index": child_index
                    }
                })
                child_index += 1
            continue

        # PROSE
        if hasattr(element, 'text') and element.text.strip():
            prose_text = element.text.strip()
            text_lower = prose_text.lower()

            # Check for section boundary in text content
            is_boundary = any(
                phrase in text_lower
                for phrase in SECTION_BOUNDARY_PHRASES
            )

            if is_boundary:
                # Check hardcoded first
                hardcoded = None
                for phrase, section in HARDCODED_BOUNDARIES.items():
                    if phrase in text_lower:
                        hardcoded = section
                        break
                
                if hardcoded:
                    current_section = hardcoded
                    continue
                
                # Fallback to LLM
                detected = detect_section(prose_text)
                if detected != "General":
                    current_section = detected
                    continue

            # Normal prose
            parent_id = str(uuid.uuid4())
            parents[parent_id] = {
                "content": prose_text,
                "metadata": {
                    **filing_metadata,
                    "chunk_type": "prose",
                    "section": current_section,
                    "page": page_no
                }
            }

            sentences = split_into_sentences(prose_text)
            for i in range(0, len(sentences), 2):
                pair = sentences[i:i + 2]
                child_text = " ".join(pair)
                if child_text.strip():
                    children.append({
                        "content": child_text,
                        "metadata": {
                            **filing_metadata,
                            "parent_id": parent_id,
                            "chunk_type": "prose",
                            "section": current_section,
                            "page": page_no,
                            "child_index": child_index
                        }
                    })
                    child_index += 1

    return parents, children


print("✓ SECTION_BOUNDARY_PHRASES defined at module level")
print("✓ create_parent_child_chunks() updated")

✓ SECTION_BOUNDARY_PHRASES defined at module level
✓ create_parent_child_chunks() updated


In [8]:
# Clear cache — force reclassification with improved prompt
section_cache.clear()
print(f"Cache cleared: {len(section_cache)} entries")

# Re-run chunking for all filings
all_parents = {}
all_children = {}

for filing in FILINGS:
    ticker = filing["ticker"]
    result = all_results[ticker]["result"]
    filing_metadata = all_results[ticker]["metadata"]

    print(f"\nChunking {filing['company']}...")
    parents, children = create_parent_child_chunks(
        ticker, result, filing_metadata
    )

    all_parents[ticker] = parents
    all_children[ticker] = children

    sections = Counter(c["metadata"]["section"] for c in children)
    print(f"  Parents : {len(parents)}")
    print(f"  Children: {len(children)}")
    print(f"  Sections:")
    for section, count in sections.most_common():
        print(f"    {section}: {count}")

total_parents = sum(len(p) for p in all_parents.values())
total_children = sum(len(c) for c in all_children.values())
print(f"\nTotal parents : {total_parents}")
print(f"Total children: {total_children}")

Cache cleared: 0 entries

Chunking Apple...
  Parents : 192
  Children: 434
  Sections:
    Notes: 97
    Risk Factors: 76
    MD&A: 52
    Legal Proceedings: 46
    General: 39
    Income Statement: 37
    Balance Sheet: 30
    Cash Flow: 28
    Shareholders Equity: 17
    Signature: 7
    Exhibits: 5

Chunking Microsoft...
  Parents : 460
  Children: 990
  Sections:
    Income Statement: 342
    Risk Factors: 265
    MD&A: 171
    Cash Flow: 76
    General: 38
    Balance Sheet: 34
    Notes: 34
    Shareholders Equity: 24
    Legal Proceedings: 3
    Exhibits: 2
    Signature: 1

Total parents : 652
Total children: 1424


In [9]:
os.makedirs("../data/processed", exist_ok=True)

docstore = {}
for ticker, parents in all_parents.items():
    docstore.update(parents)

with open("../data/processed/docstore.json", "w") as f:
    json.dump(docstore, f)

print(f"✓ Docstore saved: {len(docstore)} parents")

✓ Docstore saved: 652 parents


In [10]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

INDEX_NAME = "financial-reconciliation"
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536
BATCH_SIZE = 100

existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Creating index: {INDEX_NAME}")
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("✓ Index created")
else:
    print(f"✓ Index already exists: {INDEX_NAME}")

index = pc.Index(INDEX_NAME)
print(f"✓ Connected to index: {INDEX_NAME}")
print(f"\nCurrent stats:")
print(index.describe_index_stats())

Creating index: financial-reconciliation
✓ Index created
✓ Connected to index: financial-reconciliation

Current stats:
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [11]:
def get_embeddings(texts: list) -> list:
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [e.embedding for e in response.data]


def scale_dense(vector: list, alpha: float) -> list:
    return [v * alpha for v in vector]


def scale_sparse(vector: dict, alpha: float) -> dict:
    return {
        "indices": vector["indices"],
        "values": [v * (1 - alpha) for v in vector["values"]]
    }


def upsert_children(ticker: str, children: list):
    namespace = f"{ticker}_PC"
    print(f"Upserting {ticker} → namespace: {namespace}")
    print(f"Total children: {len(children)}")

    skipped = 0
    for i in tqdm(range(0, len(children), BATCH_SIZE)):
        batch = children[i:i + BATCH_SIZE]
        texts = [c["content"] for c in batch]

        dense_embeddings = get_embeddings(texts)
        sparse_embeddings = bm25_encoder.encode_documents(texts)

        vectors = []
        for child, dense, sparse in zip(batch, dense_embeddings, sparse_embeddings):
            if not sparse["values"]:
                skipped += 1
                continue

            chunk_id = make_chunk_id(ticker, child["metadata"]["child_index"])
            vectors.append({
                "id": chunk_id,
                "values": dense,
                "sparse_values": sparse,
                "metadata": {
                    **child["metadata"],
                    "content": child["content"][:1000]
                }
            })

        if vectors:
            index.upsert(vectors=vectors, namespace=namespace)

    print(f"  ✓ Done | skipped: {skipped} empty sparse vectors")


# Fit BM25 on all children
all_child_texts = []
for ticker, children in all_children.items():
    for child in children:
        all_child_texts.append(child["content"])

print(f"Fitting BM25 on {len(all_child_texts)} children...")
bm25_encoder = BM25Encoder()
bm25_encoder.fit(all_child_texts)
print("✓ BM25 fitted")

# Clear existing PC namespaces and re-upsert
print("\nClearing existing namespaces...")
for ticker in ["AAPL", "MSFT"]:
    try:
        index.delete(delete_all=True, namespace=f"{ticker}_PC")
        print(f"  ✓ {ticker}_PC cleared")
    except:
        print(f"  {ticker}_PC — nothing to clear")

# Upsert all children
print("\nUpserting children...")
for ticker, children in all_children.items():
    upsert_children(ticker, children)

print("\nFinal index stats:")
print(index.describe_index_stats())

Fitting BM25 on 1424 children...


  0%|          | 0/1424 [00:00<?, ?it/s]

✓ BM25 fitted

Clearing existing namespaces...
  AAPL_PC — nothing to clear
  MSFT_PC — nothing to clear

Upserting children...
Upserting AAPL → namespace: AAPL_PC
Total children: 434


100%|██████████| 5/5 [00:20<00:00,  4.08s/it]


  ✓ Done | skipped: 2 empty sparse vectors
Upserting MSFT → namespace: MSFT_PC
Total children: 990


100%|██████████| 10/10 [00:37<00:00,  3.77s/it]


  ✓ Done | skipped: 1 empty sparse vectors

Final index stats:
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'AAPL_PC': {'vector_count': 432},
                'MSFT_PC': {'vector_count': 989}},
 'total_vector_count': 1421,
 'vector_type': 'dense'}


In [12]:
bm25_encoder.dump("../data/processed/bm25_encoder.json")
print("✓ BM25 encoder saved")

✓ BM25 encoder saved


In [13]:
class QueryRoute(BaseModel):
    chunk_type: Literal["table", "prose"]
    section: Literal[
        "Income Statement",
        "Balance Sheet",
        "Cash Flow",
        "MD&A",
        "Risk Factors",
        "Legal Proceedings",
        "Notes",
        "Shareholders Equity",
        "Any"
    ]

def route_query(query: str) -> dict:
    system_prompt = """You are a financial document query router for SEC 10-Q filings.

Given a user question, return chunk_type and section.


PRIORITY RULES — apply these first:
- Question contains "inventor" (inventory/inventories) → ALWAYS Balance Sheet, table
- Question asks about dividends PAID as cash outflow → ALWAYS Cash Flow, table
- Question asks for R&D as percentage or ratio → ALWAYS MD&A, prose
- Question asks for EPS computation or weighted average shares → ALWAYS Notes, table
- "gross margin" as a dollar figure → Income Statement, table
- "R&D expense" or "research and development expense" as dollar amount → Income Statement, table  
- "operating expenses" as dollar figure → Income Statement, table
- "selling general and administrative" as dollar figure → Income Statement, table

IMPORTANT: -Questions asking WHAT WAS [metric] → table
           -Questions asking WHY DID [metric] change → prose + MD&A
           - Questions with 'change', 'grew', 'increased', 'decreased' about a metric → MD&A, prose
             UNLESS asking for the raw dollar figure with 'what was' → Income Statement, table

SECTION GUIDE:
- "Income Statement": revenue, net sales, gross margin, operating income, 
   net income, cost of sales, R&D dollar amount, selling general administrative,
   operating expenses, total operating expenses
- "Balance Sheet": total assets, cash, liabilities, equity total, receivables, inventory
- "Cash Flow": operating activities, investing, financing, cash generated, dividends paid, capex
- "MD&A": why metrics changed, segment performance, growth reasons, percentage change, ratios
- "Risk Factors": risks, uncertainties, challenges
- "Legal Proceedings": lawsuits, investigations, legal matters
- "Notes": share repurchase details, RSUs, debt details, EPS computation, weighted average shares
- "Shareholders Equity": retained earnings change, AOCI, equity statement details
- "Any": spans multiple sections

CHUNK TYPE RULES:
- Specific NUMBER or FIGURE → table
- EXPLANATION, REASON, WHY, HOW → prose
- Even if question includes "compare" → classify by PRIMARY subject
- Percentage change, growth rate → MD&A, prose

Return ONLY valid JSON matching the schema."""

    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        response_format=QueryRoute,
        temperature=0
    )

    route = response.choices[0].message.parsed
    return {"chunk_type": route.chunk_type, "section": route.section}


# Test
test_queries = [
    "What was Apple total net sales in Q2 2026?",
    "Why did Apple iPhone revenue increase?",
    "What was Apple total assets?",
    "What was Apple diluted EPS Q2 2026?",
    "How much did Apple pay in dividends?"
]

print("Query Router test:\n")
for q in test_queries:
    route = route_query(q)
    print(f"Q: {q}")
    print(f"   → {route}\n")

Query Router test:

Q: What was Apple total net sales in Q2 2026?
   → {'chunk_type': 'table', 'section': 'Income Statement'}

Q: Why did Apple iPhone revenue increase?
   → {'chunk_type': 'prose', 'section': 'MD&A'}

Q: What was Apple total assets?
   → {'chunk_type': 'table', 'section': 'Balance Sheet'}

Q: What was Apple diluted EPS Q2 2026?
   → {'chunk_type': 'table', 'section': 'Notes'}

Q: How much did Apple pay in dividends?
   → {'chunk_type': 'table', 'section': 'Cash Flow'}



In [14]:
NOISE_SECTIONS = ["General", "Signature", "Exhibits"]

def retrieve_with_parent_child(
    query: str,
    ticker: str,
    section: str = None,
    chunk_type: str = None,
    top_k: int = 5,
    alpha: float = 0.5
) -> list:

    namespace = f"{ticker}_PC"
    dense_vector = get_embeddings([query])[0]
    sparse_vector = bm25_encoder.encode_queries(query)

    # Build filter
    filter_dict = {"section": {"$nin": NOISE_SECTIONS}}
    if section and section != "Any":
        filter_dict["section"] = {"$eq": section}
    if chunk_type:
        filter_dict["chunk_type"] = {"$eq": chunk_type}

    results = index.query(
        vector=scale_dense(dense_vector, alpha),
        sparse_vector=scale_sparse(sparse_vector, alpha),
        top_k=top_k * 3,
        namespace=namespace,
        include_metadata=True,
        filter=filter_dict
    )

    seen_parents = set()
    retrieved = []

    for match in results.matches:
        parent_id = match.metadata.get("parent_id")
        if not parent_id or parent_id in seen_parents:
            continue
        seen_parents.add(parent_id)
        parent = docstore.get(parent_id)
        if parent:
            retrieved.append({
                "child_score": match.score,
                "child_content": match.metadata.get("content"),
                "parent_content": parent["content"],
                "metadata": {**match.metadata, "parent_id": parent_id}
            })
        if len(retrieved) >= top_k:
            break

    return retrieved


def retrieve_with_router(query: str, ticker: str, top_k: int = 5, alpha: float = 0.5):
    route = route_query(query)
    section = route["section"]
    chunk_type = route["chunk_type"]

    retrieved = retrieve_with_parent_child(
        query=query,
        ticker=ticker,
        section=section,
        chunk_type=chunk_type,
        top_k=top_k,
        alpha=alpha
    )
    return retrieved, route


# Quick test
query = "What was Apple total net sales in Q2 2026?"
results, route = retrieve_with_router(query, "AAPL", top_k=3)

print(f"Query : {query}")
print(f"Route : {route}")
print(f"Results: {len(results)}\n")
for i, r in enumerate(results, 1):
    print(f"Result {i} | score: {r['child_score']:.4f} | section: {r['metadata'].get('section')}")
    print(f"  Child: {r['child_content'][:100]}")

Query : What was Apple total net sales in Q2 2026?
Route : {'chunk_type': 'table', 'section': 'Income Statement'}
Results: 2

Result 1 | score: 0.2611 | section: Income Statement
  Child: |                                              | Three Months Ended   | Three Months Ended   | Six M
Result 2 | score: 0.2334 | section: Income Statement
  Child: |                                                                              | Three Months Ended 


In [15]:
GOLDEN_QUERIES = [
    # Income Statement — AAPL
    {"id": "Q001", "query": "What was Apple total net sales in Q2 2026?", "ticker": "AAPL", "expected_section": "Income Statement", "expected_chunk_type": "table"},
    {"id": "Q002", "query": "What was Apple net income in Q2 2026?", "ticker": "AAPL", "expected_section": "Income Statement", "expected_chunk_type": "table"},
    {"id": "Q003", "query": "What was Apple gross margin in Q2 2026?", "ticker": "AAPL", "expected_section": "Income Statement", "expected_chunk_type": "table"},
    {"id": "Q004", "query": "What was Apple research and development expense Q2 2026?", "ticker": "AAPL", "expected_section": "Income Statement", "expected_chunk_type": "table"},
    {"id": "Q005", "query": "What was Apple diluted earnings per share Q2 2026?", "ticker": "AAPL", "expected_section": "Notes", "expected_chunk_type": "table"},
    # Balance Sheet — AAPL
    {"id": "Q006", "query": "What was Apple total assets in Q2 2026?", "ticker": "AAPL", "expected_section": "Balance Sheet", "expected_chunk_type": "table"},
    {"id": "Q007", "query": "What was Apple cash and cash equivalents Q2 2026?", "ticker": "AAPL", "expected_section": "Balance Sheet", "expected_chunk_type": "table"},
    # MD&A — AAPL
    {"id": "Q008", "query": "By what percentage did Apple total net sales grow in Q2 2026?", "ticker": "AAPL", "expected_section": "MD&A", "expected_chunk_type": "prose"},
    {"id": "Q009", "query": "Why did Apple iPhone net sales increase in Q2 2026?", "ticker": "AAPL", "expected_section": "MD&A", "expected_chunk_type": "prose"},
    {"id": "Q010", "query": "What was Apple Greater China net sales change in Q2 2026?", "ticker": "AAPL", "expected_section": "MD&A", "expected_chunk_type": "table"},
    # Cash Flow — AAPL
    {"id": "Q011", "query": "What was Apple cash generated by operating activities in H1 2026?", "ticker": "AAPL", "expected_section": "Cash Flow", "expected_chunk_type": "table"},
    {"id": "Q012", "query": "How much did Apple pay in dividends during H1 2026?", "ticker": "AAPL", "expected_section": "Cash Flow", "expected_chunk_type": "table"},
    # Notes — AAPL
    {"id": "Q013", "query": "What was Apple share repurchase activity in Q2 2026?", "ticker": "AAPL", "expected_section": "Notes", "expected_chunk_type": "prose"},
    # Income Statement — MSFT
    {"id": "Q014", "query": "What was Microsoft total revenue in Q1 FY2026?", "ticker": "MSFT", "expected_section": "Income Statement", "expected_chunk_type": "table"},
    {"id": "Q015", "query": "What was Microsoft net income in Q1 FY2026?", "ticker": "MSFT", "expected_section": "Income Statement", "expected_chunk_type": "table"},
]

print(f"Golden queries: {len(GOLDEN_QUERIES)}")
sections = Counter(q["expected_section"] for q in GOLDEN_QUERIES)
tickers = Counter(q["ticker"] for q in GOLDEN_QUERIES)
print("\nBy section:")
for s, c in sections.most_common():
    print(f"  {s}: {c}")
print("\nBy ticker:")
for t, c in tickers.most_common():
    print(f"  {t}: {c}")

Golden queries: 15

By section:
  Income Statement: 6
  MD&A: 3
  Notes: 2
  Balance Sheet: 2
  Cash Flow: 2

By ticker:
  AAPL: 13
  MSFT: 2


In [16]:
def evaluate_retrieval(golden_queries: list, top_k: int = 5, alpha: float = 0.5) -> tuple:
    results = []

    for gq in golden_queries:
        matches, route = retrieve_with_router(
            query=gq["query"],
            ticker=gq["ticker"],
            top_k=top_k,
            alpha=alpha
        )

        hit = False
        reciprocal_rank = 0
        relevant_count = 0

        for rank, match in enumerate(matches, 1):
            section_match = match["metadata"].get("section") == gq["expected_section"]
            type_match = match["metadata"].get("chunk_type") == gq["expected_chunk_type"]
            is_relevant = section_match and type_match

            if is_relevant:
                relevant_count += 1
                if not hit:
                    hit = True
                    reciprocal_rank = 1 / rank

        results.append({
            "id": gq["id"],
            "query": gq["query"],
            "ticker": gq["ticker"],
            "expected_section": gq["expected_section"],
            "expected_type": gq["expected_chunk_type"],
            "routed_section": route["section"],
            "routed_type": route["chunk_type"],
            "hit": hit,
            "reciprocal_rank": reciprocal_rank,
            "precision_at_k": relevant_count / top_k
        })

    hit_rate = sum(r["hit"] for r in results) / len(results)
    mrr = sum(r["reciprocal_rank"] for r in results) / len(results)
    avg_precision = sum(r["precision_at_k"] for r in results) / len(results)

    return results, {
        "hit_rate": round(hit_rate, 4),
        "mrr": round(mrr, 4),
        "avg_precision_at_k": round(avg_precision, 4),
        "top_k": top_k,
        "total_queries": len(results)
    }


# Run evaluation
print("Running evaluation...")
results, metrics = evaluate_retrieval(GOLDEN_QUERIES, top_k=5, alpha=0.5)

print("\n" + "="*50)
print("FINAL METRICS — Clean notebook")
print("="*50)
print(f"Hit Rate@{metrics['top_k']}     : {metrics['hit_rate']}")
print(f"MRR              : {metrics['mrr']}")
print(f"Avg Precision@{metrics['top_k']} : {metrics['avg_precision_at_k']}")
print(f"Total queries    : {metrics['total_queries']}")

print("\nPer query breakdown:")
print("-"*70)
for r in results:
    status = "✓" if r["hit"] else "✗"
    routing = "→" if r["routed_section"] == r["expected_section"] else "✗route"
    print(f"{status} {r['id']} | {r['ticker']} | {r['expected_section']:<20} | routed={r['routed_section']:<20} {routing} | RR={r['reciprocal_rank']:.2f}")

Running evaluation...

FINAL METRICS — Clean notebook
Hit Rate@5     : 0.8667
MRR              : 0.8667
Avg Precision@5 : 0.48
Total queries    : 15

Per query breakdown:
----------------------------------------------------------------------
✓ Q001 | AAPL | Income Statement     | routed=Income Statement     → | RR=1.00
✓ Q002 | AAPL | Income Statement     | routed=Income Statement     → | RR=1.00
✓ Q003 | AAPL | Income Statement     | routed=Income Statement     → | RR=1.00
✓ Q004 | AAPL | Income Statement     | routed=Income Statement     → | RR=1.00
✓ Q005 | AAPL | Notes                | routed=Notes                → | RR=1.00
✓ Q006 | AAPL | Balance Sheet        | routed=Balance Sheet        → | RR=1.00
✓ Q007 | AAPL | Balance Sheet        | routed=Balance Sheet        → | RR=1.00
✓ Q008 | AAPL | MD&A                 | routed=MD&A                 → | RR=1.00
✓ Q009 | AAPL | MD&A                 | routed=MD&A                 → | RR=1.00
✗ Q010 | AAPL | MD&A                 | routed=M

In [18]:
# Check actual counts using describe_index_stats
stats = index.describe_index_stats()
print("Pinecone namespace stats:")
print(stats)

# Also query specifically for Income Statement
results_is = index.query(
    vector=get_embeddings(["total net sales revenue income"])[0],
    top_k=10,
    namespace="AAPL_PC",
    include_metadata=True,
    filter={"section": {"$eq": "Income Statement"}}
)
print(f"\nAAPL Income Statement chunks found: {len(results_is.matches)}")
for m in results_is.matches[:3]:
    print(f"  score={m.score:.4f} | {m.metadata.get('content', '')[:80]}")

Pinecone namespace stats:
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'AAPL_PC': {'vector_count': 432},
                'MSFT_PC': {'vector_count': 989}},
 'total_vector_count': 1421,
 'vector_type': 'dense'}

AAPL Income Statement chunks found: 10
  score=0.4559 | |                                              | Three Months Ended   | Three Mo
  score=0.4349 | |                                              | Three Months Ended   | Three Mo
  score=0.4308 | |                                                                              |
